# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">

## Setup
Install the libraries before running the lesson and tasks. The Yelp dataset is included in `dataset/07-yelp-dataset.txt`. Both models use the same held-out test set, and their text representations are fitted on training reviews only.

In [ ]:
%pip install -q pandas scikit-learn gensim torch

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [2]:
from pathlib import Path
import pandas as pd

dataset_path = Path("dataset/07-yelp-dataset.txt")
if not dataset_path.is_file():
    raise FileNotFoundError("Run this notebook from lab6 with its dataset folder present.")

df = pd.read_csv(dataset_path, sep="\t", names=["review", "label"])
if df[["review", "label"]].isna().any().any():
    raise ValueError("The dataset contains missing reviews or labels.")
if set(df["label"].unique()) != {0, 1}:
    raise ValueError("Expected binary labels: 0 (negative) and 1 (positive).")
print("Number of reviews:", len(df))
df.head()

Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [3]:
print(df["label"].value_counts())

label
1    500
0    500
Name: count, dtype: int64


## 2. Preprocessing the Dataset

In [4]:
import re

def preprocess_text(text):
    # 1. Lowercase.
    text = text.lower()
    # Keep negation in common contractions before removing punctuation.
    text = re.sub(r"n['’]t\b", " not", text)
    # 2. Replace punctuation and special characters with spaces.
    text = re.sub(r"[^a-z\s]", " ", text)
    # 3. Normalize spaces.
    return re.sub(r"\s+", " ", text).strip()

df["review"] = df["review"].apply(preprocess_text)
df = df[df["review"].str.len() > 0].copy()
# Keep identical cleaned reviews out of both sides of the train/test split.
labels_per_review = df.groupby("review")["label"].transform("nunique")
df = df[labels_per_review == 1].drop_duplicates(subset=["review"]).copy()
print("Nonempty, unique reviews:", len(df))
df.head()

Nonempty, unique reviews: 995


,review,label
0,wow loved this place,1
1,crust is not good,0
2,not tasty and the texture was just nasty,0
3,stopped by during the late may bank holiday of...,1
4,the selection on the menu was great and so wer...,1


## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [5]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 796
Testing reviews: 199


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (796, 1000)
Testing TF-IDF shape: (199, 1000)


## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [7]:
import torch
import torch.nn as nn

# Use a reproducible initialization and modest CPU parallelism.
torch.manual_seed(42)
torch.set_num_threads(2)

X_train_tensor = torch.tensor(X_train_tfidf.toarray(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_tfidf.toarray(), dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)

## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [8]:
input_size = X_train_tensor.shape[1]
# torch.size([800, 1000])

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [9]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [10]:
epochs = 10
baseline_losses = []
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()  # Clear gradients before every update.
    outputs = model(X_train_tensor)
    loss = loss_function(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    baseline_losses.append(loss.item())
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 5/10 - Loss: 0.6263
Epoch 10/10 - Loss: 0.3609


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")

Test Accuracy: 0.754


## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [12]:
print("Classification Report:")
print(classification_report(
    y_test, predictions, labels=[0, 1], target_names=["Negative", "Positive"], zero_division=0
))
print("Confusion Matrix (rows: actual; columns: predicted):")
print(confusion_matrix(y_test, predictions, labels=[0, 1]))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.80      0.68      0.73        99
    Positive       0.72      0.83      0.77       100

    accuracy                           0.75       199
   macro avg       0.76      0.75      0.75       199
weighted avg       0.76      0.75      0.75       199

Confusion Matrix (rows: actual; columns: predicted):
[[67 32]
 [17 83]]


## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


### Tokenize the Reviews and Train Word2Vec
The reviews already contain cleaned text, so split on whitespace to obtain word tokens. Train Skip-Gram Word2Vec on the training reviews only, using the four parameters required by the task.

In [13]:
import numpy as np
from gensim.models import Word2Vec

train_review_tokens = [review.split() for review in X_train]
test_review_tokens = [review.split() for review in X_test]

word2vec_model = Word2Vec(
    sentences=train_review_tokens,
    vector_size=200,
    window=5,
    min_count=1,
    sg=1,
    workers=1,
    seed=42,
    epochs=5,
)
print("Word2Vec vocabulary size:", len(word2vec_model.wv))
print("Word vector dimensions:", word2vec_model.vector_size)

Word2Vec vocabulary size: 1770
Word vector dimensions: 200


### Convert Each Review to One Vector
Average the word vectors in each review. Ignore test words absent from the training vocabulary. A review with no known words receives a zero vector. This averaging gives a fixed 200-dimensional input but loses word order.

In [14]:
def review_vector(tokens, model):
    known_words = [word for word in tokens if word in model.wv]
    if not known_words:
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean(model.wv[known_words], axis=0).astype(np.float32)

train_review_vectors = np.stack([
    review_vector(tokens, word2vec_model) for tokens in train_review_tokens
])
test_review_vectors = np.stack([
    review_vector(tokens, word2vec_model) for tokens in test_review_tokens
])

test_token_count = sum(len(tokens) for tokens in test_review_tokens)
known_test_token_count = sum(
    word in word2vec_model.wv for tokens in test_review_tokens for word in tokens
)
print("Training vectors:", train_review_vectors.shape)
print("Test vectors:", test_review_vectors.shape)
print(f"Test token vocabulary coverage: {known_test_token_count / test_token_count:.1%}")

Training vectors: (796, 200)
Test vectors: (199, 200)
Test token vocabulary coverage: 88.0%


### Convert Review Vectors to PyTorch Tensors

In [15]:
X_train_word2vec = torch.tensor(train_review_vectors, dtype=torch.float32)
X_test_word2vec = torch.tensor(test_review_vectors, dtype=torch.float32)
y_train_word2vec = torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)

print("Training input tensor:", X_train_word2vec.shape)
print("Test input tensor:", X_test_word2vec.shape)
print("Training label tensor:", y_train_word2vec.shape)

Training input tensor: torch.Size([796, 200])
Test input tensor: torch.Size([199, 200])
Training label tensor: torch.Size([796, 1])


## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

### Build the Required Network
Use three hidden layers of 128, 64, and 32 neurons, each followed by ReLU, and one output neuron. The input is the 200-dimensional average Word2Vec vector. Leave the output as a logit for `BCEWithLogitsLoss`.

In [16]:
torch.manual_seed(42)
word2vec_network = nn.Sequential(
    nn.Linear(200, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
)
print(word2vec_network)

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)


### Train for 15 Epochs
Use binary cross-entropy with logits and Adam at learning rate `0.001`. Train in shuffled batches of 32 reviews, clear gradients for each update, and record the average training loss for each epoch. Do not use test labels during training.

In [17]:
from torch.utils.data import DataLoader, TensorDataset

training_dataset = TensorDataset(X_train_word2vec, y_train_word2vec)
training_loader = DataLoader(
    training_dataset, batch_size=32, shuffle=True,
    generator=torch.Generator().manual_seed(42),
)
word2vec_loss_function = nn.BCEWithLogitsLoss()
word2vec_optimizer = torch.optim.Adam(word2vec_network.parameters(), lr=0.001)
word2vec_losses = []

for epoch in range(15):
    word2vec_network.train()
    total_loss = 0.0
    for batch_reviews, batch_labels in training_loader:
        word2vec_optimizer.zero_grad()
        train_logits = word2vec_network(batch_reviews)
        training_loss = word2vec_loss_function(train_logits, batch_labels)
        training_loss.backward()
        word2vec_optimizer.step()
        total_loss += training_loss.item() * len(batch_reviews)
    average_loss = total_loss / len(training_dataset)
    word2vec_losses.append(average_loss)
    print(f"Epoch {epoch + 1}/15 - Loss: {average_loss:.4f}")

Epoch 1/15 - Loss: 0.6949
Epoch 2/15 - Loss: 0.6938
Epoch 3/15 - Loss: 0.6937
Epoch 4/15 - Loss: 0.6938
Epoch 5/15 - Loss: 0.6933


Epoch 6/15 - Loss: 0.6933


Epoch 7/15 - Loss: 0.6937


Epoch 8/15 - Loss: 0.6936


Epoch 9/15 - Loss: 0.6935
Epoch 10/15 - Loss: 0.6934
Epoch 11/15 - Loss: 0.6934
Epoch 12/15 - Loss: 0.6935


Epoch 13/15 - Loss: 0.6936


Epoch 14/15 - Loss: 0.6932


Epoch 15/15 - Loss: 0.6933


### Calculate Test Accuracy
Switch to evaluation mode, disable gradient calculations, and apply sigmoid to obtain probabilities. Predict positive sentiment when the probability is at least `0.5`. Report the measured accuracy without assuming that Word2Vec will outperform TF-IDF.

In [18]:
word2vec_network.eval()
with torch.no_grad():
    test_logits = word2vec_network(X_test_word2vec)
    word2vec_probabilities = torch.sigmoid(test_logits)
    word2vec_predictions = (word2vec_probabilities >= 0.5).int().numpy().ravel()

word2vec_accuracy = accuracy_score(y_test, word2vec_predictions)
print(f"Word2Vec neural network test accuracy: {word2vec_accuracy:.3f}")

Word2Vec neural network test accuracy: 0.503


### Interpret the Recorded Results
The saved run achieved **75.4% test accuracy with TF-IDF** and **50.3% with the Word2Vec network**. The Word2Vec result is close to chance on this balanced dataset. Its training loss also remained near 0.693, indicating weak learning. A small training corpus and averaging word vectors may limit the representation; stronger performance would require further experiments beyond the required architecture and 15 epochs. The test set was kept separate from vocabulary fitting and model training.